# Notebook 04 — Model Comparison

Load results from `results/metrics/all_model_results.json` and produce comparison tables and figures.

In [ ]:
import sys
sys.path.insert(0, '..')
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.evaluation import model_comparison_table
from src.visualization import plot_confusion_matrix, plot_model_comparison
from src.data_loader import AAMI_CLASSES
plt.style.use('dark_background')

with open('../results/metrics/all_model_results.json') as f:
    results = json.load(f)

comp_df = model_comparison_table(results)
print(comp_df[['Accuracy','Balanced Acc','Macro F1','Weighted F1']].to_string())

In [ ]:
fig = plot_model_comparison(comp_df, metric='Macro F1')
plt.savefig('../results/figures/nb_model_f1.png', dpi=120, bbox_inches='tight')
plt.show()

# Best model confusion matrix
with open('../results/metrics/best_model.json') as f:
    best_name = json.load(f)['best_model']
print(f'Best model: {best_name}')

cm = results[best_name]['confusion_matrix']
fig = plot_confusion_matrix(cm, title=f'Best Model: {best_name}')
plt.savefig('../results/figures/nb_best_cm.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class F1 heatmap
pc_rows = []
for model_name, res in results.items():
    row = {'Model': model_name}
    for cls in AAMI_CLASSES:
        row[f'F1-{cls}'] = res.get('per_class', {}).get(cls, {}).get('f1', 0)
    pc_rows.append(row)
pc_df = pd.DataFrame(pc_rows).set_index('Model')
print('\nPer-class F1 scores:')
print(pc_df.round(4).to_string())